# Image Embedding Visualization Pipeline (v0 – TensorBoard Projector)

## Intro

This notebook reconstructs the first working version of an image embedding visualization pipeline using TensorBoard Projector.

It is intentionally kept minimal and “close to the metal” to understand:
 - how image embeddings are extracted using a pretrained CNN
 - how embeddings must be aligned with metadata and sprite images
 - why deterministic ordering is critical
 - how TensorBoard Projector consumes static files

**Important context**

This version is not production-ready. It has known limitations:
 - assumes flat or semi-flat folder structure
 - limited dataset robustness
 - manual file path handling
 - no dataset normalization pipeline

These issues are later solved in:
```
scripts/download_data.py
scripts/prepare_data.py
scripts/build_projector.py
```

**Goal of this notebook**

Build the minimal working pipeline:
```
images → ResNet18 → embeddings → TSV + sprite + metadata → TensorBoard Projector
```

## Step 1 - Build feature extraction model

We use a pretrained ResNet18 model from torchvision and remove the classification head.

In [12]:
from pathlib import Path
from PIL import Image
import torch
import torchvision.models as models
import csv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = models.ResNet18_Weights.DEFAULT
base_model = models.resnet18(weights=weights)

# Remove classification layer → output = 512-d embedding
model = torch.nn.Sequential(*list(base_model.children())[:-1])
model.to(device)
model.eval()

# Use official preprocessing pipeline (important for correctness)
transform = weights.transforms()

## Step 2 - Deterministic image ordering

TensorBoard Projector requires strict alignment between:
 - embeddings
 - metadata
 - sprite image

So we enforce deterministic ordering:

In [13]:
image_paths = sorted(Path("../images").rglob("*.jpeg"))

## Step 3 - Extract feature vectors

Each image is converted into a 512-dimensional embedding.

In [14]:
def get_vector(path: Path):
    with Image.open(path) as img:
        img = img.convert("RGB")
        batch = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            vec = model(batch)

    return vec.squeeze().cpu().numpy()

vecs = [get_vector(p) for p in image_paths]

with open("../vis/feature_vecs.tsv", "w") as fw:
    csv.writer(fw, delimiter="\t").writerows(vecs)

## Step 4 - Create sprite image

TensorBoard uses a sprite sheet to display thumbnails in the embedding space.

In [15]:
import numpy as np

images = []
for p in image_paths:
    with Image.open(p) as img:
        images.append(img.resize((100, 100)))

w, h = images[0].size
grid = int(np.ceil(np.sqrt(len(images))))

sprite = Image.new("RGB", (w * grid, h * grid))

for idx, img in enumerate(images):
    row, col = divmod(idx, grid)
    sprite.paste(img, (col * w, row * h))

sprite.save("../vis/sprite.jpg")

## Step 5 - Create metadata file

Metadata links each point in embedding space to its class label.

In [16]:
from pathlib import Path
import csv

image_paths = sorted(Path("../images").rglob("*.jpeg"))

with open("../vis/metadata.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(["cat_id", "pid"])  # header

    for p in image_paths:
        cat_id = p.parent.name          # folder name (cat, dog, ...)
        pid = p.stem                   # file name without extension (cat_1)
        writer.writerow([cat_id, pid])

## Step 6 - TensorBoard Projector config

This file tells TensorBoard how to load embeddings.

Create:
`vis/projector_config.pbtxt`

```
embeddings {
  tensor_name: "resnet18_embeddings"
  tensor_path: "feature_vecs.tsv"
  metadata_path: "metadata.tsv"

  sprite {
    image_path: "sprite.jpg"
    single_image_dim: [100, 100]
  }
}
```

## Step 7 - Run TensorBoard

```
tensorboard --logdir ./vis
```

Then open:
```
http://localhost:6006
```


## Key learning insights from this version

This “v0 pipeline” reveals important constraints:
 - embeddings, metadata, and sprite must be perfectly aligned
 - ordering is critical (sorting is not optional)
 - preprocessing must match model expectations
 - TensorBoard Projector is file-driven, not dynamic


## Summary

This notebook represents the starting point of the system before refactoring into a modular pipeline.

It is kept for:
 - educational clarity
 - debugging reference
 - understanding TensorBoard internals